In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("merging.csv"); df.head()

,Unnamed: 0,text,label
0,0,I think the issue of elderly drivers is a comp...,1
1,1,I agree that television violence has a negativ...,1
2,2,I understand the concerns of people who are op...,1
3,3,"Dear Principal, I am writing to you today to e...",1
4,4,"Yes, there is a cause that I actively support:...",1


In [3]:
x = df['text']
y = df['label']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 8000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post')

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 256, input_length=max_len),
    LSTM(128),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X_train_pad, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)


Epoch 1/15


60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 141ms/step - accuracy: 0.6795 - loss: 0.5475 - val_accuracy: 0.7426 - val_loss: 0.4443
Epoch 2/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 135ms/step - accuracy: 0.7878 - loss: 0.3688 - val_accuracy: 0.7658 - val_loss: 0.6127
Epoch 3/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 127ms/step - accuracy: 0.7936 - loss: 0.3401 - val_accuracy: 0.7827 - val_loss: 0.3812
Epoch 4/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 132ms/step - accuracy: 0.7957 - loss: 0.2813 - val_accuracy: 0.8038 - val_loss: 0.3394
Epoch 5/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 135ms/step - accuracy: 0.7936 - loss: 0.3081 - val_accuracy: 0.7954 - val_loss: 0.3344
Epoch 6/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 10s 136ms/step - accuracy: 0.7957 - loss: 0.2814 - val_accuracy: 0.7869 - val_loss: 0.3890
Epoch 7/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 133ms/step - accuracy: 0.8073 - loss: 0.2777 - val_accuracy: 0.7890 - val_loss: 0.3781
Epoch 8/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 8s 132ms/step - accuracy: 0.8126 - loss: 0.2749 - val_accuracy: 0.7806 - va

In [15]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=1)
print("Test Accuracy:", acc)


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.7909 - loss: 0.3519
Test Accuracy: 0.7908937335014343


In [16]:
def predict_text(text):
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')
    prob = model.predict(seq)[0][0]
    return prob, ("AI" if prob > 0.5 else "Human")

print(predict_text("After removing those models, what remains?You’ll mostly have:Text ML / DL classifiersTextBlob sentimentSpam detectionSimple vision (digits, emotion)Audio emotion (CPU-based)Recommendation (non-SVD)Lightweight CNN / classical MLThis becomes a light-to-medium ML Flask app"))
print(predict_text('Hi there iam Sufiyan and hows going on?'))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step
(np.float32(0.49977624), 'Human')
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
(np.float32(0.49977615), 'Human')


In [ ]:
model.save("deeplearning_version2_0.keras")


In [14]:
import pickle

with open("tokenizer_version.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
